In [1]:
from typing import Any, List, Callable, Union
import h5py
import pickle
from pathlib import Path
import numpy as np
from scipy.stats import pearsonr
import torch
from enformer_pytorch import Enformer
from enformer_pytorch import from_pretrained
from enformer_pytorch.finetune import HeadAdapterWrapper
from transformers import get_scheduler
from torch.utils.data import TensorDataset, DataLoader

from torch.optim import AdamW
from tqdm.auto import tqdm

import sys
sys.path.append('../src')
import enformer_dataloader_np
from enformer_dataloader_np import NumpyDataModule

# use GPU 1:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

/home/rajesh/projects/hackathon/SAE_Hackathon/.venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Data

# Load Model

In [ ]:
#!/usr/bin/env python
import os
import argparse
import torch
import numpy as np
from tqdm import tqdm
import pytorch_lightning as pl
from enformer_pytorch import Enformer
from enformer_pytorch.finetune import HeadAdapterWrapper

# Import the NumpyDataModule class - assuming it's in your src directory
import sys
sys.path.append('src')
import enformer_dataloader_np
from enformer_dataloader_np import NumpyDataModule  # If your class definition is in this file

class EnformerWithEmbeddings(pl.LightningModule):
    def __init__(self, num_tracks=5313, target_layer='transformer.layers.5'):
        super().__init__()
        self.enformer = Enformer.from_pretrained('EleutherAI/enformer-official-rough')
        self.model = HeadAdapterWrapper(
            enformer=self.enformer,
            num_tracks=num_tracks,
            post_transformer_embed=False
        )
        
        self.target_layer = target_layer
        self.hook_store = {}
        self.set_hooks()
        
    def set_hooks(self):
        """Set up hooks to capture embeddings from target layer"""
        def get_activation(name):
            def hook(module, input, output):
                self.hook_store[name] = output.detach()
            return hook
        
        # Navigate to the target layer and register the hook
        layer_parts = self.target_layer.split('.')
        target = self.model.enformer
        for part in layer_parts:
            target = getattr(target, part)
        
        target.register_forward_hook(get_activation(self.target_layer))
        
    def get_embeddings(self, x):
        """Extract embeddings from the target layer and also return model predictions
        
        Returns:
            tuple: (embeddings, predictions) where embeddings are from the target layer
                  and predictions are the full model output
        """
        self.eval()
        with torch.no_grad():
            # Run a forward pass to trigger the hooks and get predictions
            predictions = self.model(x)
            # Return the captured embeddings and predictions
            return self.hook_store[self.target_layer], predictions
    
    def forward(self, x, target=None):
        preds = self.model(x)
        if target is None:
            return preds
        return self.model(seq=x, target=target)

# part of forward pass of SAE training
def process_dataset(data_loader, model, output_dir, device='cuda', max_batches=None):
    """Process all batches in a dataset and save embeddings to disk"""
    os.makedirs(output_dir, exist_ok=True)
    
    batch_count = 0
    with torch.no_grad():
        for batch_idx, (sequences, targets) in enumerate(tqdm(data_loader, desc="Processing Batches")):
            if max_batches is not None and batch_idx >= max_batches:
                break
                
            # Move data to device
            sequences = sequences.to(device)
            
            # Get embeddings and predictions
            embeddings, predictions = model.get_embeddings(sequences)
            
            # Save embeddings and predictions for each sample in the batch
            for i in range(sequences.shape[0]):
                # Create a unique identifier for this sample
                sample_id = batch_idx * data_loader.batch_size + i
            
            batch_count += 1
    
    print(f"Processed {batch_count} batches, saved embeddings to {output_dir}")


human_data = '/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_npz/human'
mouse_data = '/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_npz/mouse'
            
output_dir = '/home/rajesh/projects/hackathon/SAE_Hackathon/results/enformer_embeddings'
target_layer = 'conv_tower.5.2.to_attn_logits'
batch_size = 4
max_batches = None
gpu = True  

# Set device
device = 'cuda' if gpu and torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Initialize the model
target_layer = 'conv_tower.5.2.to_attn_logits'
model = EnformerWithEmbeddings(target_layer=target_layer)
model = model.to(device)
model.eval()

# Create output directories
human_output_dir = os.path.join(output_dir, 'human')
mouse_output_dir = os.path.join(output_dir, 'mouse')

# Initialize data modules
human_data_module = NumpyDataModule(
    train_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human/train",
    val_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human/valid",
    test_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human/test",
    batch_size=batch_size
)

mouse_data_module = NumpyDataModule(
    train_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/mouse/train",
    val_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/mouse/valid",
    test_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/mouse/test",
    batch_size=batch_size
)

# Setup data modules
human_data_module.setup()
mouse_data_module.setup()

# Get data loaders
human_loader = human_data_module.test_dataloader()
mouse_loader = mouse_data_module.test_dataloader()

# Process human dataset
print("Processing human dataset...")
process_dataset(human_loader, model, human_output_dir, device, max_batches)

# Process mouse dataset
print("Processing mouse dataset...")
process_dataset(mouse_loader, model, mouse_output_dir, device, max_batches)

print("Embedding extraction complete!")


def train_sae(config):
    """
    Train a vanilla Sparse Autoencoder on Enformer embeddings

    Args:
        config: Dictionary with training configuration
            - data_dir: Path to embeddings directory
            - species: 'human' or 'mouse'
            - input_dim: Dimension of Enformer embeddings
            - hidden_dim: Dimension of sparse layer
            - l1_coefficient: L1 sparsity penalty
            - tied_weights: Whether to use tied weights
            - batch_size: Batch size for training
            - learning_rate: Learning rate for optimizer
            - weight_decay: Weight decay for optimizer
            - max_epochs: Maximum number of training epochs
            - patience: Early stopping patience
            - output_dir: Directory to save model and results
            - seed: Random seed for reproducibility
            - num_workers: Number of dataloader workers

    Returns:
        best_model: The trained SAE model
        trainer: PyTorch Lightning trainer
    """
    import os
    import torch
    import numpy as np
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
    from pytorch_lightning.loggers import TensorBoardLogger
    
    # Import our vanilla SAE implementation
    from vanilla_sae import VanillaSAE, EnformerEmbeddingDataModule
    
    # Set random seed for reproducibility
    pl.seed_everything(config["seed"])
    
    # Create output directory
    os.makedirs(config["output_dir"], exist_ok=True)
    
    # Create data module
    data_module = EnformerEmbeddingDataModule(
        data_dir=config["data_dir"],
        species=config["species"],
        batch_size=config["batch_size"],
        num_workers=config["num_workers"]
    )
    
    # Set up the data
    data_module.setup()
    
    # Create model
    model = VanillaSAE(
        input_dim=config["input_dim"],
        hidden_dim=config["hidden_dim"],
        l1_coefficient=config["l1_coefficient"],
        tied_weights=config["tied_weights"],
        learning_rate=config["learning_rate"],
        weight_decay=config["weight_decay"]
    )
    
    # Set up callbacks
    checkpoint_callback = ModelCheckpoint(
        dirpath=os.path.join(config["output_dir"], "checkpoints"),
        filename="{epoch}-{val_loss:.4f}",
        monitor="val_loss",
        save_top_k=3,
        mode="min"
    )
    
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=config["patience"],
        mode="min"
    )
    
    # Set up logger
    logger = TensorBoardLogger(
        save_dir=os.path.join(config["output_dir"], "logs"),
        name=config.get("experiment_name", "vanilla_sae")
    )
    
    # Create trainer
    trainer = pl.Trainer(
        max_epochs=config["max_epochs"],
        callbacks=[checkpoint_callback, early_stopping],
        logger=logger,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        log_every_n_steps=10,
        deterministic=True
    )
    
    # Train the model
    trainer.fit(model, data_module)
    
    # Test the model
    trainer.test(model, data_module)
    
    # Load the best model for analysis
    best_model_path = checkpoint_callback.best_model_path
    if best_model_path:
        print(f"Loading best model from {best_model_path}")
        best_model = VanillaSAE.load_from_checkpoint(best_model_path)
    else:
        best_model = model
    
    return best_model, trainer


# Load SAE

In [5]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl

from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
# from pytorch_lightning.loggers import WandbLogger
# import wandb

# If your `NumpyDataModule` is in src/enformer_dataloader_np.py
sys.path.append("src")
import enformer_dataloader_np
from enformer_dataloader_np import NumpyDataModule

# -------------------------------------------------------------------------
# 1) Enformer Wrapper for Embeddings
# -------------------------------------------------------------------------
from enformer_pytorch import Enformer
from enformer_pytorch.finetune import HeadAdapterWrapper

class EnformerWithEmbeddings(pl.LightningModule):
    """
    Wraps the Enformer model and captures a specific layer's output.
    """
    def __init__(self, num_tracks=5313, target_layer='transformer.layers.5'):
        super().__init__()
        # Pretrained Enformer
        self.enformer = Enformer.from_pretrained('EleutherAI/enformer-official-rough')
        
        # HeadAdapter to keep the shape consistent with existing code
        self.model = HeadAdapterWrapper(
            enformer=self.enformer,
            num_tracks=num_tracks,
            post_transformer_embed=False
        )
        
        self.target_layer = target_layer
        self.hook_store = {}
        self._set_hooks()

    def _set_hooks(self):
        """Register a forward hook on the user-specified target layer."""
        def get_activation(name):
            def hook(module, inp, out):
                # Detach to avoid storing gradients
                self.hook_store[name] = out.detach()
            return hook
        
        # Navigate to the target layer
        layer_parts = self.target_layer.split('.')
        target = self.model.enformer
        for part in layer_parts:
            target = getattr(target, part)
        
        # Register forward hook
        target.register_forward_hook(get_activation(self.target_layer))

    @torch.no_grad()
    def get_embeddings(self, x):
        """
        Forward pass through Enformer to populate hook_store with target layer embeddings.
        Returns (embeddings, predictions).
        """
        _ = self.model(x)  # triggers forward hooks
        # Retrieve the stored embeddings
        emb = self.hook_store[self.target_layer]
        return emb, _

    def forward(self, x):
        """
        You could call this directly, but typically we'll just use get_embeddings().
        """
        return self.model(x)

# -------------------------------------------------------------------------
# 2) Sparse Autoencoder
# -------------------------------------------------------------------------
class SparseAutoencoder(nn.Module):
    """
    A basic MLP-based autoencoder with an L1 penalty on the hidden activations
    to encourage sparsity.
    """
    def __init__(self, input_dim, hidden_dim=256, l1_weight=1e-5):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.l1_weight = l1_weight

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, input_dim),
            # If embeddings are unbounded, no final activation. 
            # If they're e.g. in [0,1], you might use nn.Sigmoid().
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z  # return reconstruction and hidden representation

# -------------------------------------------------------------------------
# 3) Combined LightningModule to Train AE on-the-fly w/ Enformer
# -------------------------------------------------------------------------
class OnTheFlyAETrainer(pl.LightningModule):
    """
    This module:
      - Uses EnformerWithEmbeddings to get embeddings on-the-fly.
      - Passes them to SparseAutoencoder.
      - Computes MSE + L1 sparse penalty on the hidden layer.
    """
    def __init__(
        self,
        enformer_target_layer="conv_tower.5.2.to_attn_logits",
        input_dim=128,
        hidden_dim=256,
        l1_weight=1e-5,
        lr=1e-3
    ):
        super().__init__()
        # Save hyperparams for logging
        self.save_hyperparameters()

        # 1) Enformer for embeddings
        self.enformer_model = EnformerWithEmbeddings(
            target_layer=enformer_target_layer
        )

        # 2) Sparse Autoencoder
        self.ae = SparseAutoencoder(
            input_dim=input_dim, 
            hidden_dim=hidden_dim, 
            l1_weight=l1_weight
        )

        # 3) Learning rate
        self.lr = lr

    def forward(self, x):
        """
        Not typically used directly by Lightning for training_step,
        but we define it for completeness:
          - x: one batch of sequences
          - returns x_hat (reconstruction of embeddings)
        """
        with torch.no_grad():
            emb, _ = self.enformer_model.get_embeddings(x)
        # Flatten if necessary (depending on shape).
        emb = emb.view(emb.size(0), -1)

        x_hat, z = self.ae(emb)
        return x_hat, z

    def training_step(self, batch, batch_idx):
        """
        Called by Lightning. We'll:
          1) Extract on-the-fly embeddings from Enformer.
          2) Run them through the autoencoder.
          3) Compute MSE + L1 penalty.
        """
        sequences, _ = batch
        # 1) Extract embeddings
        with torch.no_grad():
            emb, _ = self.enformer_model.get_embeddings(sequences)
        emb = emb.view(emb.size(0), -1)

        # 2) Forward pass in AE
        x_hat, z = self.ae(emb)

        # 3) Compute loss
        recon_loss = F.mse_loss(x_hat, emb)
        l1_loss = z.abs().mean()  # L1 penalty on hidden
        loss = recon_loss + self.ae.l1_weight * l1_loss

        # Logging
        self.log("train_recon_loss", recon_loss, prog_bar=True)
        self.log("train_l1_loss", l1_loss, prog_bar=True)
        self.log("train_loss", loss, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):
        sequences, _ = batch
        with torch.no_grad():
            emb, _ = self.enformer_model.get_embeddings(sequences)
        emb = emb.view(emb.size(0), -1)

        x_hat, z = self.ae(emb)
        recon_loss = F.mse_loss(x_hat, emb)
        l1_loss = z.abs().mean()
        loss = recon_loss + self.ae.l1_weight * l1_loss

        self.log("val_recon_loss", recon_loss, prog_bar=True)
        self.log("val_l1_loss", l1_loss, prog_bar=True)
        self.log("val_loss", loss, prog_bar=True)

        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

# -------------------------------------------------------------------------
# 4) Example Usage in a Jupyter Notebook
# -------------------------------------------------------------------------
def run_training_jupyter():
    """
    Example function showing how to set up everything in a single notebook cell.
    Adjust paths and hyperparameters to your needs.
    """
    # Initialize wandb logger
    # wandb_logger = WandbLogger(project="enformer-sparse-autoencoder", log_model=True)

    # Create data module from your NPZ partitioned data
    data_module = NumpyDataModule(
        train_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human/train",
        val_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human/valid",
        test_files_pattern="/home/rajesh/projects/hackathon/SAE_Hackathon/data/enformer_data_partitioned/human/test",
        batch_size=4,
        num_workers=2
    )
    data_module.setup()

    # Initialize the OnTheFly model
    model = OnTheFlyAETrainer(
        enformer_target_layer="conv_tower.5.2.to_attn_logits",
        input_dim=768,      # e.g. if flattening ends up as 768
        hidden_dim=256,
        l1_weight=1e-4,
        lr=1e-3
    )

    # Callbacks
    checkpoint_cb = ModelCheckpoint(
        monitor="val_loss",
        mode="min",
        save_top_k=3,
        dirpath="checkpoints/",
        filename="sae-{epoch:02d}-{val_loss:.4f}"
    )
    early_stop_cb = EarlyStopping(
        monitor="val_loss",
        patience=5,
        mode="min"
    )

    # Create trainer
    trainer = pl.Trainer(
        max_epochs=10,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=1,
        logger=wandb_logger,
        callbacks=[checkpoint_cb, early_stop_cb],
        deterministic=True,
        log_every_n_steps=5
    )

    # Train
    trainer.fit(model, data_module)

    # Optionally test
    trainer.test(model, data_module)

    # Finish wandb run (if desired)
    wandb.finish()

    # Return trainer, model if further analysis is needed
    return trainer, model


run_training_jupyter()


ModuleNotFoundError: No module named 'enformer_dataloader_np'

In [2]:
!pwd


/bin/sh: line 1: uv: command not found


# Evaluate Model

# Interpret model